# Exercise Sheet 10

This markdown block is used to define macros for later markdown blocks. 
$\newcommand{\normaldist}{\mathcal{N}}$
$\newcommand{\betadist}{\text{Beta}}$
$\newcommand{\reals}{\mathbb{R}}$
$\newcommand{\ML}{\text{ML}}$
$\renewcommand{\vec}[1]{\boldsymbol{\mathbf{#1}}}$
$\newcommand{\matrix}[1]{\boldsymbol{\mathbf{#1}}}$
$\newcommand{\dataset}{\mathcal{D}}$
$\newcommand{\class}{\mathcal{C}}$
$\newcommand{\optimal}[1]{#1^{\star}}$
$\newcommand{\argmin}{\text{argmin}}$
$\newcommand{\argmax}{\text{argmax}}$
$\newcommand{\expct}{\mathbb{E}}$
$\newcommand{\entropy}[1]{H[#1]}$
$\newcommand{\conditionalentropy}[2]{\entropy{#1 | #2}}$
$\newcommand{\kldiv}[2]{KL(#1 || #2)}$
$\newcommand{\mutualinfo}[2]{I[#1;#2]}$
$\newcommand{\deriv}[2]{\frac{d}{d #2} \left( #1\right)}$
$\newcommand{\inputs}{\matrix{X}}$
$\newcommand{\identitymtx}{\matrix{I}}$
$\newcommand{\designmtx}{\matrix{\Phi}}$
$\newcommand{\featurevec}{\boldsymbol{\phi}}$
$\newcommand{\weights}{\vec{w}}$
$\newcommand{\inputvec}{\vec{x}}$
$\newcommand{\norm}[1]{\lVert #1 \rVert}$
$\newcommand{\grad}{\nabla}$

The code in this notebook has been adapted from [NN from scratch](https://github.com/pangolulu/neural-network-from-scratch).

## Exercise 10.1 Basics of Neural Networks

Have a look at the 2-layer network equation on the slide **Basic Neural Network: Complete** and convince yourself that it is describing the network visualised on the slide **Network Diagrams**.

### 10.1 a)
Consider an arbitrary hidden unit in a 2 layer network. Let's call this $z_{j}$ for some $j= 1 \ldots M$. Where would this be drawn in  the diagram on slide **Network Diagrams**? What input nodes $x_{i}$ would connect to this node and what weights would be associated with those connections?

### 10.1 b)
What part of the equation on slide **Basic Neural Network: Complete** could be replaced by the symbol $z_{j}$? How would the equation read if this replacement was made?

### 10.1 c)
Where, if at all, would the activation $a^{(1)}_{j}$ appear in the diagram on slide **Network Diagrams**?

### 10.1 d)
What part of the equation on slide **Basic Neural Network: Complete** could be replaced by the symbol $a^{(1)}_{j}$?

## Exercise 10.2 Classification of non-linearly separable data

### 10.2 a)
The first code block below, initially generates synthetic classification data using function `sample_yin_yang` from the module `fomlads.data.synthetic_yin_yang`, then fits and plots its Fisher’s Discriminant projection.

Look at these plots and convince yourself that the data is not linearly separable.

### 10.2 b)

The second code block below fits a logistic regression model over the data, prints the misclassification error, and plots the decision boundary. Is this plot consistent with what you were expecting?

Is there a better linear decision boundary in data space than the one found here?

### 10.2 c) 

One way to improve the classification performance is to carry out some **feature engineering**. Feature engineering involves constructing feature mappings that support good performance.

In the third code block below, a new design matrix, `Phi`, is created, which is based on two RBFs. This is then used just like the data matrix to fit and plot a logistic regression classifier. Can you manipulate the parameters of the RBFs to improve upon the misclassification error from the provided code? 

*Don't worry if you cannot find a good feature mapping. The point of this is to demonstrate how difficult and/or time consuming feature engineering can be.*

You can even modify the number of RBFs, as well as the centres, and scale. Or you could explore polynomial features and see whether these perform better or worse.

*Note, you will only be able to visualise the decision regions when you have exactly **2** feature dimensions. More than 2 RBFs will preclude that here. Polynomial feature vectors of order 2 or more have more dimensions than the raw data (which here has dimension 2). As a consequence, any full polynomial feature vector will also have more dimensions than can be visualised. To address this, you can either select just two columns from a larger feature vector, or evaluate purely based on misclassification error. You also want to cut off the constant column from the design matrix as the logistic regression already adds a bias term.*

### 10.2 d)  [optional]
Imagine that the Equation on slide **Basic Neural Network: Complete** (from video part 1) applied to just a single binary target ($K=1$). Compare this with the logistic regression model (see **Unit 8** slide **Logistic Regression**):
$$p(\class_{1}|\featurevec(\vec{x})) = \sigma(\vec{w}^T \featurevec(\vec{x}) )$$

What element of the former corresponds to the feature vector $\featurevec$ in the latter? What other elements correspond between the two equations? Are there any elements of the basic neural network that have no corresponding element in logistic regression? Why not?

### 10.2 e) [optional]
Can you show that if you replaced the non-linear functions in the 2-layer network  shown in part 1 with linear functions, then the resulting model is a linear function? What model, that you have already seen on this module, is this equivalent to?

In [ ]:
## Code for 10.2 a)
import numpy as np
import matplotlib.pyplot as plt
from fomlads.model.classification import fisher_linear_discriminant_projection
from fomlads.model.classification import project_data
from fomlads.data.synthetic_yin_yang import sample_yin_yang
from fomlads.plot.exploratory import plot_class_histograms
from fomlads.plot.exploratory import plot_scatter_array_classes

N = 200
X, t = sample_yin_yang(N, noise=0.1)
plot_scatter_array_classes(X, t)

# Fisher Linear Discriminant projection
fisher_weights = fisher_linear_discriminant_projection(X, t)
fisher_X = project_data(X, fisher_weights)
plot_class_histograms(fisher_X, t)

In [ ]:
# code for 10.2 b)
from fomlads.model.basis_functions import construct_rbf_feature_mapping
from fomlads.model.basis_functions import quadratic_feature_mapping
from fomlads.model.classification import logistic_regression_fit
from fomlads.model.classification import logistic_regression_predict
from fomlads.model.classification import logistic_regression_prediction_probs
from fomlads.model.classification import construct_logistic_regression_prediction_function
from fomlads.evaluate.eval_classification import eval_accuracy
from fomlads.evaluate.eval_classification import misclassification_error
from fomlads.evaluate.eval_classification import cross_entropy_error
from fomlads.plot.predictions import plot_2d_decision_boundary


def fit_and_plot_logistic_regression(
        X, t, model_name='Logistic regression'):
    #
    weights = logistic_regression_fit(X, t)
    dim = X.shape[1]
    prediction_function = construct_logistic_regression_prediction_function(weights, add_bias_term=True)
    # only plot the decision boundary if 2 dimensional
    if dim == 2:
        plot_2d_decision_boundary(prediction_function, X, t)
    #
    predictions = prediction_function(X)
    print(f"{model_name}: missclassification error = {misclassification_error(t, predictions):.3g}")
    
    
# LOGISTIC REGRESSION FIT
fit_and_plot_logistic_regression(X, t)

In [ ]:
# 10.2 c) code - explore different choices for the centres and scale
# of RBF functions below to see if you can improve the classification
# performance by feature engineering

# 10.2 c) provided features
centres = np.array([[-1, -1.], [1, 1]])
scale = 2.0

feature_mapping = construct_rbf_feature_mapping(centres, scale)
Phi = feature_mapping(X)
fit_and_plot_logistic_regression(Phi, t)
# You can get a better fit with more centres, but this cannot be 
# visualised so easily

## Exercise 10.3


Look at the cross-entropy loss, $E(\vec{w})$, for $1$-of-$K$ classification on slide **Specific Cost Function 3**, and show that the following partial derivative holds:
$$\frac{\partial E_n}{\partial a_{nk}} = y_{nk} - t_{nk}$$

*Note that for this, $t_{nk}$, the $k$th element of the $n$th target can be treated as constant, and the $k$th output activation for the $n$th data-point is given by:*
$$a_{nk} = y_k(\vec{x}_n; \vec{w})$$
*Finally, $y_{nk}$ is the probability assigned to the $k$th class by the soft-max function:*
$$y_{nk} = \frac{e^{a_{nk}}}{\sum_{j=1}^{K} e^{a_{nj}}}$$


## 10.4 Neural network classifiers

### 10.4 a)
Now that we can perform back propagation, we can attempt to fit a neural network classifier on the data from **Ex. 10.2**. Look at the code block below, and in particular at `fit_and_plot_neural_network_classifier`. This function constructs and trains a neural network to fit the data, reporting the training error after each **epoch** (full pass through the data). The neural network is constructed using class `BasicMLP` from `fomlads.model.neural_networks` which in turn uses components of the `torch` library.

Run this code block a few times without modifying the code. Does this fit the data better than the models tried in **Ex. 10.2**? Can you explain the differences?

### 10.4 b)
The code below uses two hidden nodes to fit the data. One of the benefits of neural networks are that with enough hidden nodes they can fit any function (they are universal approximators). Try adding one or more hidden nodes and re-running the code block. Do the performance, consistency or decision boundaries change in any way?


In [ ]:
## Uncomment the line starting with % if you want interactive
## plots. You will have to install the ipympl library first.
## See: https://pypi.org/project/ipympl/
#%matplotlib widget

from fomlads.model.neural_networks import BasicMLP
from fomlads.model.neural_networks import MLP
import torch.nn as nn


def fit_and_plot_neural_network_classifier(
        X, t, layers_dim=[2, 3, 1], nonlinearity=nn.ReLU(), **trainargs):
    # by default: network has 1 input layer, 1 hidden layer, and 1 output layer
    if len(layers_dim) == 3:
        model = BasicMLP(
            layers_dim[0], layers_dim[1], layers_dim[2],
            nonlinearity=nonlinearity)
    # deeper networks are also possible
    elif len(layers_dim) > 3:
        model = MLP(layers_dim, nonlinearity=nonlinearity)
    else:
        raise ValueError(
            'layers_dim has %d values, but must have 3 or more' \
            % (len(layers_dim)))
    model.train(
        X, t, **trainargs)
    
    print(
      'Misclassification error NNs: %.3f' \
      % (misclassification_error(t, model.predict(X))))

    
# If using a different data dimension you must modify the
# input_dim parameter to the neural network
nn_input = X
input_dim = nn_input.shape[1]

# To train the neural network on the RBF input change X for Phi
# This is an unusual choice though, as the neural network hidden layers can
# be thought of as adaptive basis functions themselves. 
# WARNING! If you change the dimension of the input you need to
# update the first value of layers_dim to reflect this!
# You can experiment with different values of hidden_dim to explore
# how different sizes of hidden layer affect the predictions
# You may even want to try different numbers of layers.
hidden_dim = 2
output_dim = 1
fit_and_plot_neural_network_classifier(
    X, t, layers_dim=[input_dim, hidden_dim, output_dim],
    epochs=20000, learning_rate=0.01, reg_lambda=0.01,
    print_loss=True)

# switch off interactive plots
#plt.ioff()

## Exercise 10.5

*Solutions may not be available for this question. However, please feel free to discuss this with the module teaching staff. In particular, we would be very interested to see  any solutions you produce.*

### 10.5 a) [optional] 
You might have noticed that we didn’t explore different hyperparameter values in the code for **Ex. 10.4**. This is partly because we are essentially fitting the basis function parameters as part of the training. As such, neural networks can give reasonably good performance with a wide range of hyperparameter choices.


Nonetheless, hyperparameter tuning is still recommended, and finding the right
values is one of the greatest difficulties when working with neural networks, especially since different configurations might have unexpected effects. Increasing the
number or of layers or their dimensions could improve performance, since it increases the capacity of the neural network, but it increases the computational cost of training the network and it could also hurt the model’s performance.

Try different combinations of model parameters to get a feel for how they impact accuracy/runtime. Ideally you would want to have a separate validation set or perform cross validation for this. You may wish to explore:
* Choice of non-linear function. The code uses ReLU by default but a range of other choices are available (see the pytorch module `torch.nn` for more information)
* Learning rate
* Regularisation strength
* Number of epochs (full passes through the dataset)
* Number of nodes per layer. The input layer must equal the input dimension and the output layer the number of classes, but you can change the number of hidden nodes. You did this in **Ex. 10.4**, but this time you should think about how to make this exploration systematic.  
* Number of layers (you can modify the number of layers that make up your neural network by adding elements to the `layers_dim` list attribute. Bear in mind that adding extra layers can substantially increase computational time and model variability.

Some of these are model parameters (meaning they change the class of models you are able to fit) while others are training parameters (meaning they only influence the training characteristics). Can you identify which is which?


### 10.5 b) [optional]
You may want to see if you can  get the interactive plot working (i.e. a plot that
shows the learned decision boundary at every epoch). What can you observe in the network’s learning process? How does its performance compare with the other models you explored? Run it multiple times to convince yourself of your observations. 

*To begin look at the top of the code block for **Ex. 10.4**, but you may need to read up on the `ipympl` library [here](https://pypi.org/project/ipympl/) to get this working.*

### 10.5 c) [optional]

You could try to find some other classification datasets to experiment with using this model (and other model from the module). Remember that you will need to adjust the input dimension for other data. You could also explore how to create regression or $1$-of-$K$ classification version of the MLP shown in module `fomlads.model.neural_networks`.